# 5. Model Submission

## Objective
Load trained models and generate predictions on the test set.
Save predictions to `./submission.csv`

### Approach:
1. Load the trained T5 model
2. Stream test.csv for memory efficiency
3. Generate translations
4. Create submission file

In [ ]:
# Parameters and Paths
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import json
import re

OUTPUT_DIR = Path("output")
MODELS_DIR = OUTPUT_DIR / "models"
DATA_DIR = Path("data")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 5.1 Load Trained Model

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load the trained model
model_path = MODELS_DIR / "t5_akkadian_translator"

print(f"Loading model from {model_path}")
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)
model = model.to(device)
model.eval()

print("Model loaded successfully")

## 5.2 Text Cleaning Function (same as preprocessing)

In [ ]:
def clean_transliteration(text):
    """
    Clean Akkadian transliteration - same as in data preparation.
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Replace gaps and breaks first
    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r'\.{3,}', '<big_gap>', text)
    text = re.sub(r'\[…\s*…\]', '<big_gap>', text)
    text = re.sub(r'\[\s*\]', '', text)
    
    # Remove/modify special characters
    text = re.sub(r'[!?/:.]', '', text)
    text = re.sub(r'[˹˺]', '', text)
    text = re.sub(r'\[|\]', '', text)
    
    # Normalize Ḫ → H and ḫ → h
    text = text.replace('Ḫ', 'H').replace('ḫ', 'h')
    text = text.replace('Ṣ', 'S').replace('ṣ', 's')
    text = text.replace('Ṭ', 'T').replace('ṭ', 't')
    text = text.replace('Š', 'SZ').replace('š', 'sz')
    
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

print("Cleaning function defined")

## 5.3 Load Test Data (Streaming)

In [ ]:
# Load test data - streaming approach for large files
# For this example, we'll load it but use chunked processing

test_df = pd.read_csv(DATA_DIR / "test.csv")
print(f"Test samples: {len(test_df)}")
print(f"\nTest columns: {test_df.columns.tolist()}")
test_df.head()

## 5.4 Generate Predictions

In [ ]:
# Clean test data
test_df['transliteration_cleaned'] = test_df['transliteration'].apply(clean_transliteration)

# Generate translations
def translate_batch(texts, batch_size=16):
    """Translate a batch of texts"""
    translations = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch_texts = texts[i:i+batch_size]
        
        # Add prefix for T5
        inputs = ["translate Akkadian to English: " + text for text in batch_texts]
        
        # Tokenize
        inputs = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_length=128,
                num_beams=4,
                early_stopping=True
            )
        
        # Decode
        batch_translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(batch_translations)
    
    return translations

print("Generating translations...")
translations = translate_batch(test_df['transliteration_cleaned'].tolist(), batch_size=8)
print(f"Generated {len(translations)} translations")

In [ ]:
# Show sample translations
print("\nSample translations:")
for i in range(min(3, len(translations))):
    print(f"\n--- Sample {i} ---")
    print(f"Source: {test_df['transliteration'].iloc[i][:100]}...")
    print(f"Translation: {translations[i][:200]}...")

## 5.5 Create Submission File

In [ ]:
# Create submission dataframe
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'translation': translations
})

# Verify format matches sample submission
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("Sample submission format:")
print(sample_submission.head())

print("\nOur submission format:")
print(submission_df.head())

In [ ]:
# Save submission to ./submission.csv (root directory)
submission_path = Path("submission.csv")
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to {submission_path}")
print(f"\nSubmission shape: {submission_df.shape}")

## 5.6 Verify Submission

In [ ]:
# Verify submission file
verification_df = pd.read_csv(submission_path)
print("=== Submission Verification ===")
print(f"Rows: {len(verification_df)}")
print(f"Columns: {verification_df.columns.tolist()}")
print(f"Missing translations: {verification_df['translation'].isna().sum()}")
print(f"Empty translations: {(verification_df['translation'].str.len() == 0).sum()}")

# Show first few entries
print("\nFirst few entries:")
print(verification_df.head())

In [ ]:
# Save submission summary
submission_summary = {
    "submission_file": str(submission_path),
    "total_predictions": len(submission_df),
    "model_used": "T5-small fine-tuned on Akkadian-English",
    "model_path": str(model_path),
    "missing_translations": int(verification_df['translation'].isna().sum()),
    "empty_translations": int((verification_df['translation'].str.len() == 0).sum()),
}

with open(OUTPUT_DIR / "submission_summary.json", 'w') as f:
    json.dump(submission_summary, f, indent=2)

print("\nSubmission summary saved")
print(json.dumps(submission_summary, indent=2))

## 5.7 List All Output Files

In [ ]:
# List all output files
print("=== All Output Files ===")

print("\nOutput directory:")
for f in OUTPUT_DIR.rglob("*"):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(OUTPUT_DIR)}: {size_kb:.1f} KB")

print("\nRoot directory (submission.csv):")
for f in Path(".").glob("*.csv"):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

In [ ]:
# Verify notebook is valid JSON
notebook_path = Path("notebooks/05_model_submission.ipynb")
if notebook_path.exists():
    with open(notebook_path, 'r') as f:
        nb = json.load(f)
    print(f"Notebook is valid JSON with {len(nb['cells'])} cells")